In [2]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, confusion_matrix, accuracy_score, f1_score

In [3]:
data = pd.read_csv("loan_approval_data.csv")
data = data.drop(columns=["Applicant_ID"])
data = data.dropna(subset=["Loan_Approved"]).reset_index(drop=True)
data.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Applicant_Income    902 non-null    float64
 1   Coapplicant_Income  902 non-null    float64
 2   Employment_Status   906 non-null    object 
 3   Age                 902 non-null    float64
 4   Marital_Status      902 non-null    object 
 5   Dependents          903 non-null    float64
 6   Credit_Score        902 non-null    float64
 7   Existing_Loans      901 non-null    float64
 8   DTI_Ratio           903 non-null    float64
 9   Savings             901 non-null    float64
 10  Collateral_Value    901 non-null    float64
 11  Loan_Amount         903 non-null    float64
 12  Loan_Term           902 non-null    float64
 13  Loan_Purpose        902 non-null    object 
 14  Property_Area       901 non-null    object 
 15  Education_Level     903 non-null    object 
 16  Gender  

In [4]:
# X = data.drop(["Loan_Approved"], axis=1)
# Y = data["Loan_Approved"]

In [5]:
numerical_col = data.select_dtypes(include = ["number"]).columns
categorical_cols = [
    "Employment_Status",
    "Marital_Status",
    "Loan_Purpose",
    "Property_Area",
    "Education_Level",
    "Gender",
    "Employer_Category"
]

In [6]:
num_imp = SimpleImputer(strategy = "mean")
data[numerical_col] = num_imp.fit_transform(data[numerical_col])

In [7]:
cat_imp = SimpleImputer(strategy = "most_frequent")
data[categorical_cols] = cat_imp.fit_transform(data[categorical_cols])

In [8]:
data.isnull().sum()


Applicant_Income      0
Coapplicant_Income    0
Employment_Status     0
Age                   0
Marital_Status        0
Dependents            0
Credit_Score          0
Existing_Loans        0
DTI_Ratio             0
Savings               0
Collateral_Value      0
Loan_Amount           0
Loan_Term             0
Loan_Purpose          0
Property_Area         0
Education_Level       0
Gender                0
Employer_Category     0
Loan_Approved         0
dtype: int64

In [9]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import joblib


# Education encoder
education_encoder = LabelEncoder()
data["Education_Level"] = education_encoder.fit_transform(data["Education_Level"])

# Loan approval encoder
le = LabelEncoder()
data["Loan_Approved"] = le.fit_transform(data["Loan_Approved"])


cols = ["Employment_Status", "Marital_Status", "Loan_Purpose", "Property_Area", "Gender", "Employer_Category"]

ohe = OneHotEncoder(drop = "first", sparse_output = False, handle_unknown = "ignore")
encoded = ohe.fit_transform(data[cols])

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cols), index=data.index)

data = pd.concat([data.drop(columns=cols), encoded_df], axis=1)

In [10]:
X = data.drop(columns=["Loan_Approved"])
Y = data["Loan_Approved"]

In [11]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size = 0.2, random_state = 42
)

In [12]:
scale = StandardScaler()
X_train = scale.fit_transform(X_train)
X_test = scale.transform(X_test)

In [13]:
#Logistic Regression
log_model = LogisticRegression(max_iter = 4000)
log_model.fit(X_train, Y_train)
y_pred = log_model.predict(X_test)

print("#Logistic Regression")
print("Precision Score",precision_score(Y_test,y_pred))
print("Recall Score", recall_score(Y_test,y_pred))
print("F1_Score",f1_score(Y_test,y_pred))
print("Accuracy",accuracy_score(Y_test,y_pred))
print("CM", confusion_matrix(Y_test,y_pred))
    

#Logistic Regression
Precision Score 0.8727272727272727
Recall Score 0.7868852459016393
F1_Score 0.8275862068965517
Accuracy 0.8947368421052632
CM [[122   7]
 [ 13  48]]


In [14]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

n_neigh = [2,3,5,7,9]

for k in n_neigh:
    KNN_model = KNeighborsClassifier(n_neighbors = k)
    KNN_model.fit(X_train, Y_train)
    y_pred_KNN = KNN_model.predict(X_test)
    
    print("N_neighbors", k)
    print("#KNN")
    print("Precision Score",precision_score(Y_test,y_pred_KNN))
    print("Recall Score", recall_score(Y_test,y_pred_KNN))
    print("F1_Score",f1_score(Y_test,y_pred_KNN))
    print("Accuracy",accuracy_score(Y_test,y_pred_KNN))
    print("CM", confusion_matrix(Y_test,y_pred_KNN))
    print("\n")

N_neighbors 2
#KNN
Precision Score 0.72
Recall Score 0.29508196721311475
F1_Score 0.4186046511627907
Accuracy 0.7368421052631579
CM [[122   7]
 [ 43  18]]


N_neighbors 3
#KNN
Precision Score 0.7169811320754716
Recall Score 0.6229508196721312
F1_Score 0.6666666666666666
Accuracy 0.8
CM [[114  15]
 [ 23  38]]


N_neighbors 5
#KNN
Precision Score 0.7333333333333333
Recall Score 0.5409836065573771
F1_Score 0.6226415094339622
Accuracy 0.7894736842105263
CM [[117  12]
 [ 28  33]]


N_neighbors 7
#KNN
Precision Score 0.75
Recall Score 0.4918032786885246
F1_Score 0.594059405940594
Accuracy 0.7842105263157895
CM [[119  10]
 [ 31  30]]


N_neighbors 9
#KNN
Precision Score 0.7435897435897436
Recall Score 0.47540983606557374
F1_Score 0.58
Accuracy 0.7789473684210526
CM [[119  10]
 [ 32  29]]




In [15]:
# NAIVE BYES

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train, Y_train)
y_pred_nb = nb_model.predict(X_test)


print("#Navie Bayes")
print("Precision Score",precision_score(Y_test,y_pred_nb))
print("Recall Score", recall_score(Y_test,y_pred_nb))
print("F1_Score",f1_score(Y_test,y_pred_nb))
print("Accuracy",accuracy_score(Y_test,y_pred_nb))
print("CM", confusion_matrix(Y_test,y_pred_nb))

#Navie Bayes
Precision Score 0.8979591836734694
Recall Score 0.7213114754098361
F1_Score 0.8
Accuracy 0.8842105263157894
CM [[124   5]
 [ 17  44]]


In [16]:
import joblib

joblib.dump(log_model, "log_model.pkl")
joblib.dump(scale, "scaler.pkl")
joblib.dump(num_imp, "num_imputer.pkl")
joblib.dump(cat_imp, "cat_imputer.pkl")
joblib.dump(ohe, "onehot_encoder.pkl")

# Save both encoders
joblib.dump(education_encoder, "education_encoder.pkl")
joblib.dump(le, "label_encoder.pkl")   # Loan_Approved encoder

print("Files saved successfully!")
import sklearn
print(sklearn.__version__)

Files saved successfully!
1.7.2
